In [11]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

def get_dataloaders(batch_size=128):
    transform_train = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465),
                             (0.247, 0.243, 0.261))
    ])

    transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465),
                             (0.247, 0.243, 0.261))
    ])

    trainset = datasets.CIFAR10(
        root='./data', train=True, download=True, transform=transform_train)
    testset = datasets.CIFAR10(
        root='./data', train=False, download=True, transform=transform_test)

    trainloader = DataLoader(trainset, batch_size=batch_size,
                             shuffle=True)
    testloader = DataLoader(testset, batch_size=batch_size,
                            shuffle=False)

    return trainloader, testloader


trainloader, testloader = get_dataloaders()
print(f"Number of training batches: {len(trainloader)}")
print(f"Number of testing batches: {len(testloader)}")

for images, labels in trainloader:
  print(f"Image batch shape: {images.size()}")
  print(f"Label batch shape: {labels.size()}")
  break

Number of training batches: 391
Number of testing batches: 79
Image batch shape: torch.Size([128, 3, 32, 32])
Label batch shape: torch.Size([128])


In [12]:

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")


Using device: cuda


In [13]:
!pip install fvcore

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 3.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for fvcore: filename=fvcore-0.1.5.post20221221-py3-none-any.whl size=61397 sha256=07a9ff76ec6a0bd57b72159efb92c18a8d015bd0dd78e541c08c1fcdf40191c2
  Stored in directory: /root/.cache/pip/wheels/ed/9f/a5/e4f5b27454ccd4596bd8b62432c7d6b1ca9fa22aef9d70a16a
  Created wheel for iopath: filename=iopath-0.1.10-py3-none-any.whl size=31527 sha256=b77dd56c34e4036672a296c118ee7bd333368e4f70ac5c2f2f4afe3744b226da
  Stored in directory: /root/.cache/pip/wheels/7c/96/04/4f5f31ff812f684f69f40cb1634357812220aac58d4698048c
Successfully built fvcore iopath


In [14]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super(SimpleCNN, self).__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Linear(128 * 8 * 8, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)


In [15]:
! pip install thop

In [16]:
import torch
from thop import profile

def count_flops(model):
    # Automatically detect which device (CPU/GPU) the model is using
    device = next(model.parameters()).device
    # Create dummy input on the exact same device
    dummy_input = torch.randn(1, 3, 32, 32).to(device)

    flops, params = profile(model, inputs=(dummy_input,))
    print(f"FLOPs: {flops / 1e6:.2f} MFLOPs")
    print(f"Parameters: {params / 1e6:.2f} Million")
    return flops, params

# Initialize model and check FLOPs before training
model = SimpleCNN()
count_flops(model)

[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv2d'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.batchnorm.BatchNorm2d'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.activation.ReLU'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.pooling.MaxPool2d'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.container.Sequential'>.
[INFO] Register count_linear() for <class 'torch.nn.modules.linear.Linear'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.dropout.Dropout'>.
FLOPs: 41.26 MFLOPs
Parameters: 2.19 Million


(41257472.0, 2193674.0)

In [17]:
import matplotlib.pyplot as plt
import numpy as np
import wandb

def plot_gradient_flow(named_parameters, epoch):
    ave_grads = []
    layers = []

    for n, p in named_parameters:
        if p.requires_grad and p.grad is not None:
            layers.append(n)
            ave_grads.append(p.grad.abs().mean().item())

    plt.figure(figsize=(10, 5))
    plt.plot(ave_grads)
    plt.xticks(range(len(layers)), layers, rotation="vertical")
    plt.title(f"Gradient Flow – Epoch {epoch}")
    plt.grid(True)
    wandb.log({"Gradient Flow": wandb.Image(plt)})
    plt.close()


def plot_weight_updates(named_parameters, epoch):
    weights = []
    layers = []

    for n, p in named_parameters:
        if p.requires_grad:
            layers.append(n)
            weights.append(p.data.norm().item())

    plt.figure(figsize=(10, 5))
    plt.plot(weights)
    plt.xticks(range(len(layers)), layers, rotation="vertical")
    plt.title(f"Weight Norms – Epoch {epoch}")
    plt.grid(True)
    wandb.log({"Weight Update Flow": wandb.Image(plt)})
    plt.close()


In [18]:
import torch
import torch.nn as nn
import torch.optim as optim
import wandb

# Initialize W&B
wandb.init(
    project="CNN-CIFAR10-Lab2",
    name="SimpleCNN",
    config={
        "epochs": 30,
        "batch_size": 128,
        "optimizer": "Adam",
        "lr": 1e-3
    }
)

trainloader, testloader = get_dataloaders(batch_size=128)
model = SimpleCNN().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# Calculate FLOPs on the target device
flops, params = count_flops(model)
wandb.log({"FLOPs": flops, "Parameters": params})

for epoch in range(30):
    # --- Training Phase ---
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for i, (images, labels) in enumerate(trainloader):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()

        # Plot gradient flow once per epoch to save time
        if i == 0:
            plot_gradient_flow(model.named_parameters(), epoch)

        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    train_acc = 100. * correct / total
    avg_train_loss = running_loss / len(trainloader)

    # --- Validation Phase ---
    model.eval()
    test_loss, test_correct, test_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in testloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            test_loss += loss.item()
            _, predicted = outputs.max(1)
            test_total += labels.size(0)
            test_correct += predicted.eq(labels).sum().item()

    test_acc = 100. * test_correct / test_total
    avg_test_loss = test_loss / len(testloader)

    # Log all metrics to W&B
    wandb.log({
        "Train Loss": avg_train_loss,
        "Train Acc": train_acc,
        "Test Loss": avg_test_loss,
        "Test Acc": test_acc
    })

    # Log weight distributions
    plot_weight_updates(model.named_parameters(), epoch)

    print(f"Epoch [{epoch+1}/30] Loss: {avg_train_loss:.4f} | Acc: {train_acc:.2f}% | Test Acc: {test_acc:.2f}%")

[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv2d'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.batchnorm.BatchNorm2d'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.activation.ReLU'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.pooling.MaxPool2d'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.container.Sequential'>.
[INFO] Register count_linear() for <class 'torch.nn.modules.linear.Linear'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.dropout.Dropout'>.
FLOPs: 41.26 MFLOPs
Parameters: 2.19 Million
Epoch [1/30] Loss: 1.8108 | Acc: 32.25% | Test Acc: 47.25%
Epoch [2/30] Loss: 1.5101 | Acc: 43.48% | Test Acc: 57.67%
Epoch [3/30] Loss: 1.3987 | Acc: 48.10% | Test Acc: 61.36%
Epoch [4/30] Loss: 1.3308 | Acc: 51.03% | Test Acc: 63.77%
Epoch [5/30] Loss: 1.2721 | Acc: 53.19% | Test Acc: 66.14%
Epoch [6/30] Loss: 1.2358 | Acc: 54.68% | Test Acc: 65.81%
Epoch [7/30] Loss: 1.2081 | Acc: 55.77% | Test Acc